# Chuvas e Deslizamentos no Estado do Rio de Janeiro (2015–2024)
**Disciplina:** Linguagem de Programação — Análise e Visualização de Dados com Python · **Avaliação G1**

## 1. Introdução ao problema
O Rio de Janeiro convive com tragédias recorrentes causadas por chuvas intensas em encostas ocupadas (Região Serrana em 2011, Petrópolis em 2022). Entender **quanto a chuva explica os deslizamentos**, **onde** e **em que época do ano** eles se concentram ajuda a priorizar prevenção e resposta.

**Perguntas do projeto:** (1) a chuva é o principal fator associado às ocorrências? (2) quais regiões/municípios concentram os impactos? (3) existe sazonalidade? (4) o nível de risco classificado reflete os impactos reais?

## 2. Base de dados
Arquivo `simulacao_chuvas_deslizamentos_rj.csv`: base **simulada** com 960 linhas = 8 municípios × 120 meses (jan/2015–dez/2024).

| Coluna | Descrição |
|---|---|
| ano, mes, data | período da observação |
| municipio, regiao_rj | localização |
| populacao | população (simulada) |
| chuva_mm, temperatura_media, umidade | variáveis climáticas |
| indice_solo | índice de saturação/fragilidade do solo (0–1) |
| ocorrencias_deslizamento, desalojados, obitos | impactos |
| nivel_risco | Baixo, Médio, Alto, Crítico |

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
IMG = "../imagens"
os.makedirs(IMG, exist_ok=True)

df = pd.read_csv("../dados/simulacao_chuvas_deslizamentos_rj.csv", encoding="utf-8-sig")
print(df.shape)
df.head()

## 3. Limpeza e preparação

In [ ]:
df["data"] = pd.to_datetime(df["data"])
df["nivel_risco"] = pd.Categorical(df["nivel_risco"], ["Baixo", "Médio", "Alto", "Crítico"], ordered=True)

print("Nulos:", int(df.isna().sum().sum()), "| Duplicados:", int(df.duplicated().sum()))
print("Chave (data, município) única:", not df.duplicated(["data", "municipio"]).any())
print("Coerência ano/mês × data:", bool(((df.data.dt.year == df.ano) & (df.data.dt.month == df.mes)).all()))
df.describe().T.round(2)

**Resultado:** a base não tem nulos nem duplicados e as datas são coerentes. Observação importante: `populacao` varia de mês a mês dentro do mesmo município (ex.: Niterói aparece com mais de 5 milhões), o que é típico de simulação. Por isso **não calculamos taxas per capita** e usamos a população apenas como informação descritiva.

## 4. Engenharia de atributos

In [ ]:
df = df.sort_values(["municipio", "data"]).reset_index(drop=True)
df["periodo_chuvoso"] = df["mes"].isin([12, 1, 2, 3]).map({True: "Chuvoso (dez–mar)", False: "Menos chuvoso (abr–nov)"})
df["faixa_chuva"] = pd.cut(df["chuva_mm"], [-0.1, 50, 100, 200, 500],
                           labels=["0–50 mm", "50–100 mm", "100–200 mm", ">200 mm"])
df["chuva_acum_3m"] = df.groupby("municipio")["chuva_mm"].transform(lambda s: s.rolling(3, min_periods=1).sum())
df["houve_obito"] = (df["obitos"] > 0).astype(int)
df["desalojados_por_ocorrencia"] = (df["desalojados"] / df["ocorrencias_deslizamento"].replace(0, np.nan)).round(1)
df[["data", "municipio", "periodo_chuvoso", "faixa_chuva", "chuva_acum_3m", "houve_obito"]].head()

## 5. KPIs

In [ ]:
kpis = {
    "Ocorrências de deslizamento": int(df.ocorrencias_deslizamento.sum()),
    "Pessoas desalojadas": int(df.desalojados.sum()),
    "Óbitos": int(df.obitos.sum()),
    "Chuva média mensal (mm)": round(df.chuva_mm.mean(), 1),
    "% meses em risco Crítico": round((df.nivel_risco == "Crítico").mean() * 100, 1),
    "Município com mais ocorrências": df.groupby("municipio").ocorrencias_deslizamento.sum().idxmax(),
}
pd.Series(kpis).to_frame("valor")

## 6. Análise exploratória e gráficos

### 6.1 Sazonalidade: o regime de chuvas define o ano

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
m = df.groupby("mes")[["chuva_mm", "ocorrencias_deslizamento"]].mean().reset_index()
sns.barplot(data=m, x="mes", y="chuva_mm", color="#4C78A8", ax=ax[0])
ax[0].set(title="Chuva média por mês do ano", xlabel="Mês", ylabel="mm")
sns.barplot(data=m, x="mes", y="ocorrencias_deslizamento", color="#D1495B", ax=ax[1])
ax[1].set(title="Ocorrências médias por município-mês", xlabel="Mês", ylabel="Ocorrências")
plt.tight_layout(); plt.savefig(f"{IMG}/01_sazonalidade.png", dpi=130); plt.show()
share = df.groupby("periodo_chuvoso").ocorrencias_deslizamento.sum()
print((share / share.sum() * 100).round(1))

### 6.2 Relação entre chuva e deslizamentos

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5.5))
sns.scatterplot(data=df, x="chuva_mm", y="ocorrencias_deslizamento", hue="nivel_risco",
                palette={"Baixo": "#6BAA75", "Médio": "#E9C46A", "Alto": "#F4813F", "Crítico": "#B5232D"}, alpha=.7, ax=ax)
sns.regplot(data=df, x="chuva_mm", y="ocorrencias_deslizamento", scatter=False, color="black", line_kws={"lw": 1.5}, ax=ax)
ax.set(title="Chuva × ocorrências de deslizamento", xlabel="Chuva (mm)", ylabel="Ocorrências")
plt.tight_layout(); plt.savefig(f"{IMG}/02_chuva_vs_ocorrencias.png", dpi=130); plt.show()
print("Pearson:", round(df.chuva_mm.corr(df.ocorrencias_deslizamento), 2),
      "| Spearman:", round(df.chuva_mm.corr(df.ocorrencias_deslizamento, method="spearman"), 2))

### 6.3 Correlação estatística

In [ ]:
cols = ["chuva_mm", "chuva_acum_3m", "indice_solo", "umidade", "temperatura_media", "populacao",
        "ocorrencias_deslizamento", "desalojados", "obitos"]
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(df[cols].corr(), annot=True, fmt=".2f", cmap="RdBu_r", center=0, vmin=-1, vmax=1, ax=ax)
ax.set_title("Matriz de correlação (Pearson)")
plt.tight_layout(); plt.savefig(f"{IMG}/03_correlacao.png", dpi=130); plt.show()
df[cols].corr()["ocorrencias_deslizamento"].drop("ocorrencias_deslizamento").sort_values(ascending=False).round(2)

### 6.4 Onde está o problema: regiões e municípios

In [ ]:
mun = (df.groupby(["regiao_rj", "municipio"], as_index=False)
         .agg(ocorrencias=("ocorrencias_deslizamento", "sum"), desalojados=("desalojados", "sum"),
              obitos=("obitos", "sum"), chuva_media=("chuva_mm", "mean"))
         .sort_values("ocorrencias", ascending=False))
fig, ax = plt.subplots(1, 2, figsize=(14, 4.8))
sns.barplot(data=mun, y="municipio", x="ocorrencias", hue="regiao_rj", dodge=False, ax=ax[0])
ax[0].set(title="Ocorrências totais por município", xlabel="Ocorrências", ylabel="")
sns.barplot(data=mun, y="municipio", x="obitos", hue="regiao_rj", dodge=False, legend=False, ax=ax[1])
ax[1].set(title="Óbitos totais por município", xlabel="Óbitos", ylabel="")
plt.tight_layout(); plt.savefig(f"{IMG}/04_municipios.png", dpi=130); plt.show()
reg = df.groupby("regiao_rj")[["ocorrencias_deslizamento", "obitos"]].sum()
(reg / reg.sum() * 100).round(1).sort_values("ocorrencias_deslizamento", ascending=False)

In [ ]:
pv = df.pivot_table(index="municipio", columns="mes", values="ocorrencias_deslizamento", aggfunc="mean")
fig, ax = plt.subplots(figsize=(11, 4.5))
sns.heatmap(pv, cmap="YlOrRd", annot=True, fmt=".1f", ax=ax)
ax.set(title="Ocorrências médias: município × mês", xlabel="Mês", ylabel="")
plt.tight_layout(); plt.savefig(f"{IMG}/05_heatmap_municipio_mes.png", dpi=130); plt.show()

### 6.5 Série temporal e média móvel

In [ ]:
ts = df.groupby("data").agg(ocorrencias=("ocorrencias_deslizamento", "sum"), chuva=("chuva_mm", "mean"))
ts["mm_3m"] = ts.ocorrencias.rolling(3).mean()
ts["mm_12m"] = ts.ocorrencias.rolling(12).mean()
fig, ax = plt.subplots(figsize=(13, 4.5))
ax.plot(ts.index, ts.ocorrencias, color="#bbb", label="Mensal")
ax.plot(ts.index, ts.mm_3m, color="#F4813F", label="Média móvel 3m")
ax.plot(ts.index, ts.mm_12m, color="#B5232D", lw=2.5, label="Média móvel 12m")
ax.set(title="Ocorrências mensais no estado (soma dos 8 municípios)", ylabel="Ocorrências"); ax.legend()
plt.tight_layout(); plt.savefig(f"{IMG}/06_serie_temporal.png", dpi=130); plt.show()
anual = df.groupby("ano")[["ocorrencias_deslizamento", "obitos"]].sum(); anual

### 6.6 O nível de risco reflete os impactos?

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ordem = ["Baixo", "Médio", "Alto", "Crítico"]
sns.boxplot(data=df, x="nivel_risco", y="chuva_mm", order=ordem, color="#F4A261", ax=ax[0]); ax[0].set(title="Chuva por nível de risco", xlabel="", ylabel="mm")
ob_df = df.groupby("nivel_risco", observed=True).obitos.sum().reset_index()
sns.barplot(data=ob_df, x="nivel_risco", y="obitos", order=ordem, color="#B5232D", ax=ax[1])
ax[1].set(title="Óbitos por nível de risco", xlabel="", ylabel="Óbitos")
plt.tight_layout(); plt.savefig(f"{IMG}/07_risco.png", dpi=130); plt.show()
ob = df.groupby("nivel_risco", observed=True).obitos.sum(); (ob / ob.sum() * 100).round(1)

## 7. Persistência em banco (SQLite)

In [ ]:
import sqlite3
os.makedirs("../database", exist_ok=True)
saida = df.drop(columns=["faixa_chuva", "periodo_chuvoso"]).assign(
    nivel_risco=df.nivel_risco.astype(str), data=df.data.dt.strftime("%Y-%m-%d"))
with sqlite3.connect("../database/chuvas.db") as con:
    saida.to_sql("ocorrencias", con, if_exists="replace", index=False)
    print(pd.read_sql("SELECT municipio, SUM(ocorrencias_deslizamento) AS total FROM ocorrencias GROUP BY municipio ORDER BY total DESC", con))

## 8. Interpretação dos resultados
- **Chuva é o principal fator associado:** correlação de Pearson ≈ **0,82** entre chuva e ocorrências; o índice de solo vem em seguida (≈ 0,72). Temperatura e umidade praticamente não se relacionam (|r| < 0,05).
- **Sazonalidade forte:** dezembro a março têm cerca de 214 mm/mês contra ~97 mm nos demais meses, e concentram aproximadamente **58%** das ocorrências em apenas 4 meses.
- **Concentração geográfica:** a Região Serrana (Petrópolis, Teresópolis e Nova Friburgo) responde por ~**57%** das ocorrências e ~**61%** dos óbitos.
- **Sem tendência clara no período:** o total anual oscila entre 488 (2023) e 616 (2021), sem queda ou alta sustentada. A média móvel de 12 meses é praticamente plana.
- **Classificação de risco coerente:** meses 'Crítico' têm em média ~203 mm e respondem por cerca de **78%** dos óbitos; 'Baixo' tem zero óbitos.
- **Impacto humano acompanha a ocorrência:** desalojados e ocorrências têm correlação ≈ 0,99; óbitos são mais raros e menos previsíveis (r ≈ 0,49).

## 9. Conclusão
Os deslizamentos no RJ seguem um padrão previsível: **chuva intensa em meses de verão, sobre solos saturados, na Região Serrana**. Isso sugere que ações preventivas (alertas, limpeza de encostas, preparo de abrigos) devem ser **antecipadas para novembro–dezembro** e **priorizar Petrópolis, Teresópolis e Nova Friburgo**. Monitorar a chuva acumulada e o índice de solo é mais informativo que temperatura ou umidade.

**Limitações:** base simulada, população inconsistente no tempo, apenas 8 municípios e correlação não implica causalidade. Em estudos reais, recomenda-se integrar dados do INMET/CEMADEN/Defesa Civil.